### Day 08 · OOP 封装 (L1)

今天进入面向对象编程(OOP)的世界。
你会学会:用**类**描述事物,用**对象**承载数据,
用**方法**定义行为,用**属性**封装状态。

> **叙事锚点**:电商订单系统 ——
> 从散落的 dict 到 Product 类


**本课知识地图**

| 知识点 | 解决什么问题 |
| --- | --- |
| 类与对象 | 把相关的数据和行为打包成一个整体 |
| __init__ 构造函数 | 创建对象时自动初始化属性 |
| self 参数 | 让方法知道"当前操作的是哪个实例" |
| 实例方法 | 让对象自己拥有操作数据的能力 |
| @property getter | 把方法伪装成属性,统一访问语法 |
| @property setter | 写数据时自动校验,拒绝非法值 |
| __str__ 魔术方法 | print(对象) 输出友好可读的信息 |
| 类属性 vs 实例属性 | 区分"共享数据"与"独有数据" |
| BankAccount 综合 | 把六个知识点串成一个真实业务类 |

---


#### 类与对象 —— 图纸与产品

> **痛点**  

之前用多个变量描述一个事物(如 name, age, score),
变量散落在各处,传参时容易遗漏,
代码越长越难维护。

> **类比**  

**类** = 图纸/模具,**对象** = 按图纸造出的一个个具体产品。
比如"学生"是图纸,"张三"是一个具体的学生。

> **解释**  

`class` 关键字定义一张图纸;
用 `类名(...)` 调用图纸,得到一个**实例**(对象)。


**ASCII 内存图 —— 类与对象的关系**

```
        内存
  ┌──────────────────────┐
  │ 类对象 Student       │ ← 图纸(只有一份)
  │   __init__           │
  │   info()             │
  └──────────┬───────────┘
             │ 造出
   ┌─────────┴─────────┐
   ▼                   ▼
┌──────────┐     ┌──────────┐
│ stu1     │     │ stu2     │
│ name=张三│     │ name=李四│
│ age=18   │     │ age=20   │
└──────────┘     └──────────┘
   独立实例          独立实例
```

**关键**:类只有一份,对象可以创建无数个,
每个对象**独立**存储自己的数据。

In [ ]:
# 示例 1:定义一个最简单的类,创建两个对象
class Student:
    pass

stu1 = Student()  # 创建第一个对象
stu2 = Student()  # 创建第二个对象

print(stu1)       # <__main__.Student object at 0x...>
print(stu2)       # 地址不同 → 两个独立对象
print(stu1 is stu2)  # False(不是同一个)


**逐行解剖**

① `class Student:` 在内存中创建一个**类对象**(图纸)
② `stu1 = Student()` → 分配一块新内存 → 得到一个实例
③ `stu2 = Student()` → 再分配一块**不同**的内存 → 另一个实例
④ `stu1 is stu2` → 比较内存地址 → False(不是同一块内存)

**要点**:每次调用 `类名()` 都会创建**新**的对象,
就像用模具压铸出一个个零件,零件之间互不干扰。

In [ ]:
# 示例 2:给对象"手动"绑定属性(不推荐,先理解原理)
class Student:
    pass

stu1 = Student()
stu1.name = "张三"   # 给 stu1 绑一个 name 属性
stu1.age = 18       # 给 stu1 绑一个 age 属性

stu2 = Student()
stu2.name = "李四"
stu2.age = 20

print(stu1.name, stu1.age)  # 张三 18
print(stu2.name, stu2.age)  # 李四 20
# stu1 和 stu2 各自拥有独立的 name/age


**逐行解剖**

① `stu1 = Student()` 创建空对象
② `stu1.name = "张三"` 在 stu1 的内存里写入 name
③ `stu2 = Student()` 创建另一个空对象
④ `stu2.name = "李四"` 在 stu2 的内存里写入 name
⑤ 两个对象的 name **互不干扰**

**为什么不用这种方式**:每次都要手动写,
容易漏绑、写错。→ 引出 `__init__` 自动绑定。


**常见错误**

❌ **错误 1**:以为 stu1 和 stu2 是同一个对象
```python
stu1 = Student()
stu2 = Student()
# stu1 和 stu2 地址不同,是独立实例
```

❌ **错误 2**:忘了写括号 `()`
```python
stu1 = Student    # 错!这是把类本身赋给变量
stu1 = Student()  # 对!这才是创建对象
```


> **问自己**(先思考,再看下面的参考答案):
> - 类和对象分别对应生活中的什么?
> - 创建两个对象后,修改一个会影响另一个吗?
> - 为什么 `stu1 is stu2` 是 False?

In [ ]:
# ============ 学员代码区 ============
# 请定义一个 Product 类(暂时用 pass),
# 创建两个对象 p1、p2,
# 分别给它们绑定 title 和 price,
# 然后打印两个商品的 title。
class Product:
    pass

# p1 = Product()
# p1.title = ...
# p1.price = ...
pass

# ============ 参考答案 ============
class Product:
    pass

p1 = Product()
p1.title = "Python 入门"
p1.price = 59.8

p2 = Product()
p2.title = "算法图解"
p2.price = 45.0

print(p1.title)  # Python 入门
print(p2.title)  # 算法图解

---


#### __init__ 构造函数 —— 给对象"贴标签"

> **痛点**  

上一节手动绑属性太麻烦,容易漏写。
能不能在创建对象**的同时**就把属性绑好?

> **类比**  

买手机时填写的"配置单":
颜色、内存、价格在出厂时就定好了,
不需要用户拿到手后再贴标签。

> **解释**  

`__init__` 是**构造函数**(创建对象时自动调用),
它的参数就是创建对象时要传入的数据。


**ASCII 内存图 —— __init__ 执行过程**

```
stu1 = Student("张三", 18)

① Python 看到 Student(...) → 分配新内存
② 自动调用 __init__(self=新对象, name="张三", age=18)
③ self.name = "张三" → 新对象的 name = "张三"
④ self.age = 18     → 新对象的 age = 18
⑤ 创建完毕,把对象赋值给变量 stu1

结果:
  stu1 ──→┌───────────────┐
           │ name = "张三" │
           │ age  = 18     │
           └───────────────┘
```

**关键**:`__init__` 不需要手动调用,
Python 在 `类名(...)` 时自动触发。

In [ ]:
# 示例 1:用 __init__ 自动绑定属性
class Student:
    def __init__(self, name, age):
        self.name = name   # 把参数绑定到对象
        self.age = age     # 把参数绑定到对象

# --- 执行过程 ---
# 第 1 行 stu1 = Student("张三", 18):
#   ① Python 看到 Student(...) → 分配新内存
#   ② 自动调用 __init__(self=新对象, name="张三", age=18)
#   ③ self.name = "张三" → 新对象的 name = "张三"
#   ④ self.age = 18 → 新对象的 age = 18
#   ⑤ 创建完毕,把对象赋值给变量 stu1
#
# 第 2 行 stu2 = Student("李四", 20):
#   ① 再分配一块新内存
#   ② 自动调用 __init__(self=新对象2, name="李四", age=20)
#   ③ 新对象2 的 name = "李四", age = 20
#   ④ 赋值给 stu2
#
# 第 3 行 print(stu1.name, stu1.age):
#   ① 读取 stu1.name → "张三"
#   ② 读取 stu1.age → 18
#   ③ 输出: 张三 18

stu1 = Student("张三", 18)
stu2 = Student("李四", 20)
print(stu1.name, stu1.age)  # 张三 18
print(stu2.name, stu2.age)  # 李四 20


**逐行解剖**

① `class Student:` 定义类(图纸)
② `def __init__(self, name, age):` 定义构造函数
   - `self` 是**自动传入**的新对象本身
   - `name, age` 是创建时要传入的参数
③ `self.name = name` 把参数值**绑到对象上**
④ `stu1 = Student("张三", 18)` 触发 __init__
⑤ `print(stu1.name)` 读取对象上的属性

**命名约定**:`__init__` 前后各有**两个下划线**,
不要写成 `_init_` 或 `init`。

In [ ]:
# 示例 2:电商场景 —— Product 类
class Product:
    def __init__(self, title, price):
        self.title = title
        self.price = price

# --- 执行过程 ---
# 第 1 行 p1 = Product("Python 入门", 59.8):
#   ① 分配新内存
#   ② 调用 __init__(self=新对象, title="Python 入门", price=59.8)
#   ③ self.title = "Python 入门"
#   ④ self.price = 59.8
#   ⑤ 赋值给 p1
#
# 第 2 行 print(p1.title, p1.price):
#   ① 读取 p1.title → "Python 入门"
#   ② 读取 p1.price → 59.8
#   ③ 输出: Python 入门 59.8

p1 = Product("Python 入门", 59.8)
p2 = Product("算法图解", 45.0)
print(p1.title, p1.price)  # Python 入门 59.8
print(p2.title, p2.price)  # 算法图解 45.0


**常见错误**

❌ **错误 1**:忘记写 `self.` 前缀
```python
class Student:
    def __init__(self, name):
        name = name  # 错!这只是给局部变量赋值
        self.name = name  # 对!绑到对象上
```

❌ **错误 2**:`__init__` 写错下划线数量
```python
def _init_(self, name):   # 错!单下划线
def __init__(self, name): # 对!双下划线
```


> **问自己**:
> - `__init__` 什么时候被调用?需要手动调用吗?
> - `self.name = name` 等号左边和右边的 name 分别是什么?
> - 如果 `__init__` 漏写了 `self` 参数,会报什么错?

In [ ]:
# ============ 学员代码区 ============
# 请定义 Product 类,用 __init__ 绑定 title 和 price,
# 创建两本书 p1、p2,分别打印它们的 title 和 price。
class Product:
    pass  # 请补全 __init__

# p1 = Product("Python 入门", 59.8)
# p2 = Product("算法图解", 45.0)
pass

# ============ 参考答案 ============
class Product:
    def __init__(self, title, price):
        self.title = title
        self.price = price

p1 = Product("Python 入门", 59.8)
p2 = Product("算法图解", 45.0)
print(p1.title, p1.price)  # Python 入门 59.8
print(p2.title, p2.price)  # 算法图解 45.0

---


#### self 是什么 —— 代表"当前这个实例"

> **痛点**  

一个类可以创建很多对象,方法怎么知道
"我现在操作的是哪一个对象"?

> **类比**  

考试时每张试卷上都要写姓名。
`self` 就是试卷上的"姓名",
让老师知道"这张卷子是谁的"。

> **解释**  

`self` 是实例方法的**第一个参数**,
Python 自动把"当前对象"传进去,
你不需要(也不能)手动传。


**ASCII 内存图 —— self 的本质**

```
stu1 = Student("张三", 18)
stu2 = Student("李四", 20)

stu1.say_hi()  →  Student.say_hi(self=stu1)
stu2.say_hi()  →  Student.say_hi(self=stu2)

内存:
  ┌───────────────┐
  │ Student 类    │
  │  say_hi(self) │ ← 方法只有一个
  └──────┬────────┘
         │ 调用时 self 指向不同对象
    ┌────┴────┐
    ▼         ▼
┌─────────┐ ┌─────────┐
│ stu1    │ │ stu2    │
│name=张三│ │name=李四│
└─────────┘ └─────────┘
```

**关键**:方法定义时写 `self`,
调用时**不需要**手动传,
Python 自动把"谁调用的"传进去。

In [ ]:
# 示例 1:self 让方法知道"我在操作谁"
class Student:
    def __init__(self, name, age):
        self.name = name
        self.age = age

    def say_hi(self):
        # self 就是调用这个方法的对象
        print(f"大家好,我是{self.name},今年{self.age}岁")

# --- 执行过程 ---
# 第 1 行 stu1 = Student("张三", 18):
#   ① 创建对象,name="张三", age=18
#
# 第 2 行 stu2 = Student("李四", 20):
#   ① 创建对象,name="李四", age=20
#
# 第 3 行 stu1.say_hi():
#   ① Python 把 stu1 作为 self 传入
#   ② self.name → "张三"
#   ③ 输出: 大家好,我是张三,今年18岁
#
# 第 4 行 stu2.say_hi():
#   ① Python 把 stu2 作为 self 传入
#   ② self.name → "李四"
#   ③ 输出: 大家好,我是李四,今年20岁

stu1 = Student("张三", 18)
stu2 = Student("李四", 20)
stu1.say_hi()  # 大家好,我是张三,今年18岁
stu2.say_hi()  # 大家好,我是李四,今年20岁


**逐行解剖**

① `def say_hi(self):` 定义实例方法,**必须**有 self
② `self.name` 读取**当前对象**的 name
③ `stu1.say_hi()` 等价于 `Student.say_hi(stu1)`
④ `stu2.say_hi()` 等价于 `Student.say_hi(stu2)`

**要点**:虽然 self 可以叫别的名字(如 this),
但**强烈建议**用 self,这是 Python 社区的约定。

In [10]:
# 示例 2:验证 self 就是调用者本身
class Dog:
    def __init__(self, name):
        self.name = name

    def who_am_i(self):
        print(f"self 的 id = {id(self)}")
        print(f"我的名字是 {self.name}")

# --- 执行过程 ---
# 第 1 行 d1 = Dog("旺财"):
#   ① 创建对象,name="旺财"
#
# 第 2 行 print(f"d1 的 id = {id(d1)}"):
#   ① 输出 d1 的内存地址
#
# 第 3 行 d1.who_am_i():
#   ① self = d1(同一个对象)
#   ② 打印 self 的 id → 与 d1 的 id 相同

d1 = Dog("旺财")
print(f"d1 的 id = {id(d1)}")
d1.who_am_i()  # self 的 id 与 d1 相同

d1 的 id = 4418391408
self 的 id = 4418391408
我的名字是 旺财



**常见错误**

❌ **错误 1**:调用时手动传 self
```python
stu1.say_hi(stu1)  # 错!Python 自动传,不需要手写
stu1.say_hi()      # 对!
```

❌ **错误 2**:定义时漏写 self
```python
def say_hi():      # 错!缺少 self
    print(self.name)  # 报错: self 未定义
def say_hi(self):  # 对!
    print(self.name)
```


> **问自己**:
> - `stu1.say_hi()` 和 `stu2.say_hi()` 调用的是同一个方法吗?
> - 为什么方法定义时要写 self,调用时却不传?
> - 如果把 self 改名成 `abc`,代码还能运行吗?

In [9]:
# ============ 学员代码区 ============
# 请定义 Student 类,包含 __init__ 绑定 name,
# 并定义 introduce(self) 方法,
# 打印 "我叫XXX"。
# 创建两个学生,分别调用 introduce()。
class Student:
    pass  # 请补全

# s1 = Student("小明")
# s2 = Student("小红")
pass

# ============ 参考答案 ============
class Student:
    def __init__(self, name):
        self.name = name

    def introduce(self):
        print(f"我叫{self.name}")

s1 = Student("小明")
s2 = Student("小红")
s1.introduce()  # 我叫小明
s2.introduce()  # 我叫小红

我叫小明
我叫小红


---


#### 实例方法 —— 对象能做什么

> **痛点**  

数据绑到对象上了,但操作这些数据的函数还在外面飘着,
函数一多不知道哪个属于谁。

> **类比**  

对象是一台电视机,方法就是遥控器上的按钮。
调音量 → `tv.volume_up()`,
按钮"属于"这台电视。

> **解释**  

定义在类内部、第一个形参为 `self` 的函数
就是**实例方法**。通过 `实例.方法()` 调用。


**ASCII 内存图 —— 实例方法的调用**

```
p = Product("Python 入门", 59.8)
p.discount(0.8)

调用过程:
  ① Python 看到 p.discount(0.8)
  ② 找到 Product 类的 discount 方法
  ③ 调用 discount(self=p, rate=0.8)
  ④ self.price = 59.8 * 0.8 = 47.84
  ⑤ 返回 47.84

内存:
  p ──→ ┌────────────────┐
        │ title="Python.."│
        │ price=47.84    │ ← 被方法修改
        └────────────────┘
```

**关键**:实例方法可以**读取**和**修改**对象的数据,
这是它与普通函数最大的区别。

In [ ]:
# 示例 1:实例方法读取和修改对象数据
class Product:
    def __init__(self, title, price):
        self.title = title
        self.price = price

    def info(self):
        # 读取对象数据,返回描述字符串
        return f"商品[{self.title}] 价格:{self.price:.2f} 元"

    def discount(self, rate):
        # 修改对象数据
        self.price = self.price * rate
        return self.price

# --- 执行过程 ---
# 第 1 行 p = Product("Python 入门", 59.8):
#   ① 创建对象,title="Python 入门", price=59.8
#
# 第 2 行 print(p.info()):
#   ① self = p
#   ② 读取 self.title, self.price
#   ③ 返回 "商品[Python 入门] 价格:59.80 元"
#
# 第 3 行 print(p.discount(0.8)):
#   ① self = p, rate = 0.8
#   ② self.price = 59.8 * 0.8 = 47.84
#   ③ 返回 47.84
#
# 第 4 行 print(p.price):
#   ① 读取 p.price → 47.84(已被修改)

p = Product("Python 入门", 59.8)
print(p.info())         # 商品[Python 入门] 价格:59.80 元
print(p.discount(0.8))  # 47.84
print(p.price)          # 47.84(已被修改)


**逐行解剖**

① `def info(self):` 无额外参数,只读取数据
② `return f"..."` 返回字符串,不修改对象
③ `def discount(self, rate):` 有一个额外参数
④ `self.price = self.price * rate` **修改**对象数据
⑤ `p.discount(0.8)` 调用后,p.price 已被改变

**要点**:实例方法可以**读取**也可以**修改**对象,
修改后影响会**持久保存**在对象里。

In [ ]:
# 示例 2:电商场景 —— 计算含税价格
class Product:
    tax_rate = 0.13  # 类属性(后面会详细讲)

    def __init__(self, title, price):
        self.title = title
        self.price = price

    def price_with_tax(self):
        # 读取类属性 + 实例属性
        return self.price * (1 + Product.tax_rate)

# --- 执行过程 ---
# 第 1 行 p = Product("算法图解", 100):
#   ① 创建对象,title="算法图解", price=100
#
# 第 2 行 print(p.price_with_tax()):
#   ① self = p
#   ② 读取 self.price → 100
#   ③ 读取 Product.tax_rate → 0.13
#   ④ 计算 100 * 1.13 = 113.0

p = Product("算法图解", 100)
print(p.price_with_tax())  # 113.0


**常见错误**

❌ **错误 1**:方法内忘记写 self
```python
def info():
    return title  # 错!找不到 title
def info(self):
    return self.title  # 对!
```

❌ **错误 2**:把方法当函数调用
```python
Product.info()     # 错!缺少 self
Product.info(p)    # 对!但没人这么写
p.info()           # 对!推荐写法
```


> **问自己**:
> - 实例方法和普通函数的区别是什么?
> - 为什么实例方法可以访问对象的属性?
> - `p.info()` 和 `Product.info(p)` 效果一样吗?

In [ ]:
# ============ 学员代码区 ============
# 请定义 Product 类,包含:
# - __init__ 绑定 title 和 price
# - info(self) 返回 "商品[XXX] 价格:XX.XX 元"
# - discount(self, rate) 打折并返回新价格
# 创建一本书,调用 info() 和 discount(0.9)。
class Product:
    pass  # 请补全

# p = Product("Python 入门", 59.8)
pass

# ============ 参考答案 ============
class Product:
    def __init__(self, title, price):
        self.title = title
        self.price = price

    def info(self):
        return f"商品[{self.title}] 价格:{self.price:.2f} 元"

    def discount(self, rate):
        self.price = self.price * rate
        return self.price

p = Product("Python 入门", 59.8)
print(p.info())        # 商品[Python 入门] 价格:59.80 元
print(p.discount(0.9)) # 53.82

---


#### @property getter —— 把方法伪装成属性

> **痛点**  

有些数据需要**计算**得到(如含税价格),
如果暴露为方法,调用时要写 `p.price_with_tax()`;
如果暴露为属性,又没法计算。
能不能**像属性一样访问,但背后执行计算**?

> **类比**  

电视机的"音量"看起来是一个属性,
但背后是电路在实时计算。
`@property` 就是让方法"看起来像属性"。

> **解释**  

`@property` 装饰器把方法变成"只读属性",
访问 `obj.xxx` 时自动调用方法,无需括号。


**ASCII 内存图 —— @property 的调用**

```
class Product:
    @property
    def price_with_tax(self):
        return self.price * 1.13

p.price_with_tax  ← 没有括号!

调用过程:
  ① Python 看到 p.price_with_tax
  ② 发现它是 @property → 自动调用方法
  ③ 方法返回计算结果
  ④ 看起来就像访问属性一样

对比:
  p.price           → 直接读属性
  p.price_with_tax  → 自动调用方法(看起来也是属性)
```

**关键**:`@property` 让**方法**可以像**属性**一样访问,
不需要写括号 `()`。

In [ ]:
# 示例 1:@property 把方法伪装成属性
class Product:
    def __init__(self, title, price):
        self.title = title
        self.price = price

    @property
    def price_with_tax(self):
        # 像属性一样访问,但背后是计算
        return self.price * 1.13

# --- 执行过程 ---
# 第 1 行 p = Product("Python 入门", 100):
#   ① 创建对象,title="Python 入门", price=100
#
# 第 2 行 print(p.price_with_tax):
#   ① Python 看到 p.price_with_tax
#   ② 发现是 @property → 自动调用方法
#   ③ 计算 100 * 1.13 = 113.0
#   ④ 输出: 113.0
#
# 注意:没有写括号!

p = Product("Python 入门", 100)
print(p.price_with_tax)  # 113.0(没有括号)
print(p.price)           # 100(原始价格)


**逐行解剖**

① `@property` 装饰器,写在 `def` 上方
② `def price_with_tax(self):` 方法名就是属性名
③ `return self.price * 1.13` 返回计算结果
④ `p.price_with_tax` **没有括号**,像属性一样访问

**要点**:`@property` 是**只读**的,
不能给它赋值(赋值会报错)。

In [ ]:
# 示例 2:用 @property 做数据转换
class Student:
    def __init__(self, name, score):
        self.name = name
        self.score = score

    @property
    def level(self):
        # 根据分数返回等级
        if self.score >= 90:
            return "优秀"
        elif self.score >= 60:
            return "及格"
        else:
            return "不及格"

# --- 执行过程 ---
# 第 1 行 s = Student("小明", 85):
#   ① 创建对象,name="小明", score=85
#
# 第 2 行 print(s.level):
#   ① 自动调用 level(self=s)
#   ② 85 >= 60 → 返回 "及格"

s = Student("小明", 85)
print(s.level)  # 及格(没有括号)


**常见错误**

❌ **错误 1**:调用时加了括号
```python
p.price_with_tax()  # 错!@property 不加括号
p.price_with_tax    # 对!
```

❌ **错误 2**:试图给 @property 赋值(没有 setter)
```python
p.price_with_tax = 200
# 报错: can't set attribute
```


> **问自己**:
> - `@property` 的方法能不能有参数?
> - 为什么访问 `@property` 不需要写括号?
> - `@property` 和直接定义属性有什么区别?

In [ ]:
# ============ 学员代码区 ============
# 请定义 Student 类,包含:
# - __init__ 绑定 name 和 score
# - @property level 返回等级
#   (>=90 优秀, >=60 及格, 否则 不及格)
# 创建学生,打印 s.level。
class Student:
    pass  # 请补全

# s = Student("小明", 85)
pass

# ============ 参考答案 ============
class Student:
    def __init__(self, name, score):
        self.name = name
        self.score = score

    @property
    def level(self):
        if self.score >= 90:
            return "优秀"
        elif self.score >= 60:
            return "及格"
        else:
            return "不及格"

s = Student("小明", 85)
print(s.level)  # 及格

---


#### @property setter —— 写保护逻辑

> **痛点**  

直接把属性暴露在外面,用户可以随意赋值
负数、空串、类型错误。
比如 `acc.balance = -500` 应该被拒绝。

> **类比**  

属性像带锁的保险箱:
getter 看内容,setter 控制写入规则,
外部看起来仍是 `obj.xxx` 的写法。

> **解释**  

`@xxx.setter` 给写操作加规则,
赋值时自动调用,非法值可以拒绝。


**ASCII 内存图 —— @property 完整结构**

```
class BankAccount:
    @property
    def balance(self):       ← getter(读)
        return self._balance

    @balance.setter
    def balance(self, value): ← setter(写)
        if value < 0:
            print("拒绝")      ← 校验失败
            return
        self._balance = value  ← 校验通过,存入

内存:
  acc ──→ ┌─────────────────┐
          │ _balance = 1000  │ ← 真实数据存在 _balance
          └─────────────────┘
           acc.balance → 自动调用 getter → 1000
           acc.balance = -500 → 自动调用 setter → 拒绝
```

**关键**:
- getter 负责**读**,setter 负责**写**
- 真实数据存在 `_balance`(下划线开头=私有)
- 外部仍用 `acc.balance` 访问,语法不变

In [ ]:
# 示例 1:@property + setter 保护余额
class BankAccount:
    def __init__(self, owner, balance):
        self.owner = owner
        self.balance = balance  # 走 setter 校验

    @property
    def balance(self):
        # getter:读取时自动调用
        return self._balance

    @balance.setter
    def balance(self, value):
        # setter:赋值时自动调用
        if value < 0:
            print("余额不能为负,已忽略")
            return
        self._balance = value

# --- 执行过程 ---
# 第 1 行 acc = BankAccount("张三", 1000):
#   ① 创建对象
#   ② self.balance = 1000 → 触发 setter
#   ③ 1000 >= 0 → self._balance = 1000
#
# 第 2 行 print(acc.balance):
#   ① 触发 getter → 返回 self._balance → 1000
#
# 第 3 行 acc.balance = -500:
#   ① 触发 setter,value = -500
#   ② -500 < 0 → 打印"余额不能为负",拒绝
#
# 第 4 行 print(acc.balance):
#   ① 触发 getter → 1000(没变)
#
# 第 5 行 acc.balance = 2000:
#   ① 触发 setter,value = 2000
#   ② 2000 >= 0 → self._balance = 2000

acc = BankAccount("张三", 1000)
print(acc.balance)    # 1000
acc.balance = -500    # 余额不能为负,已忽略
print(acc.balance)    # 1000(没变)
acc.balance = 2000
print(acc.balance)    # 2000


**逐行解剖**

① `@property` 装饰 getter 方法
② `def balance(self):` getter 方法名 = 属性名
③ `return self._balance` 返回**私有**属性
④ `@balance.setter` 装饰 setter 方法
⑤ `def balance(self, value):` 接收要赋的值
⑥ `if value < 0:` 校验逻辑
⑦ `self._balance = value` 通过校验后存入

**命名约定**:
getter/setter 方法名相同(都是 `balance`),
真实数据存在 `_balance`(单下划线=内部使用)。

In [ ]:
# 示例 2:保护学生成绩(0~100)
class Student:
    def __init__(self, name, score):
        self.name = name
        self.score = score  # 走 setter

    @property
    def score(self):
        return self._score

    @score.setter
    def score(self, value):
        if value < 0 or value > 100:
            print("成绩必须在 0-100 之间")
            return
        self._score = value

# --- 执行过程 ---
# 第 1 行 s = Student("小明", 85):
#   ① self.score = 85 → 触发 setter
#   ② 0 <= 85 <= 100 → self._score = 85
#
# 第 2 行 s.score = 150:
#   ① 触发 setter,value = 150
#   ② 150 > 100 → 打印错误,拒绝
#
# 第 3 行 print(s.score):
#   ① 触发 getter → 85(没变)

s = Student("小明", 85)
s.score = 150  # 成绩必须在 0-100 之间
print(s.score)  # 85(没变)


**常见错误**

❌ **错误 1**:setter 里直接写 `self.score = value`
```python
@score.setter
def score(self, value):
    self.score = value  # 错!会无限递归!
    self._score = value # 对!用 _score 存储
```

❌ **错误 2**:getter 和 setter 名称不一致
```python
@property
def score(self): ...    # getter 叫 score

@grade.setter           # 错!应该是 @score.setter
def score(self, v): ...
```


> **问自己**:
> - 为什么真实数据要存在 `_balance` 而不是 `balance`?
> - setter 里如果写 `self.balance = value` 会怎样?
> - 如果没有 setter,能给 @property 赋值吗?

In [ ]:
# ============ 学员代码区 ============
# 请定义 Student 类,用 @property 保护 score:
# - getter 返回当前成绩
# - setter 禁止超出 0~100,
#   非法时打印 "成绩必须在 0-100 之间"
# 创建学生,尝试赋值为 150,再打印 score。
class Student:
    pass  # 请补全

# s = Student("小明", 85)
pass

# ============ 参考答案 ============
class Student:
    def __init__(self, name, score):
        self.name = name
        self.score = score

    @property
    def score(self):
        return self._score

    @score.setter
    def score(self, value):
        if value < 0 or value > 100:
            print("成绩必须在 0-100 之间")
            return
        self._score = value

s = Student("小明", 85)
s.score = 150  # 成绩必须在 0-100 之间
print(s.score)  # 85(没变)

---


#### __str__ 魔术方法 —— print(对象) 输出友好信息

> **痛点**  

`print(对象)` 默认打印
`<__main__.Student object at 0x7f8b1c>`,
调试时完全看不出对象里装了什么数据。

> **类比**  

每个人的"自我介绍":
别人问你"你是谁",你说"我叫张三,今年 18 岁",
而不是说"我是人类,住在地球某处"。

> **解释**  

`__str__` 是**魔术方法**,
`print()` / `str()` 调用对象时自动触发,
返回你定义的"友好字符串"。


**ASCII 内存图 —— __str__ 的调用**

```
没有 __str__:
  print(stu) → <__main__.Student object at 0x...>

有 __str__:
  class Student:
      def __str__(self):
          return f"Student(name={self.name})"

  print(stu) → Student(name=张三)

调用过程:
  ① Python 看到 print(stu)
  ② 自动调用 stu.__str__()
  ③ 返回字符串
  ④ print 输出这个字符串
```

**关键**:`__str__` 必须**返回**字符串,
不能在里面直接 print。

In [ ]:
# 示例 1:没有 __str__ vs 有 __str__
class Student:
    def __init__(self, name, age):
        self.name = name
        self.age = age

# --- 执行过程 ---
# 第 1 行 stu = Student("张三", 18):
#   ① 创建对象,name="张三", age=18
#
# 第 2 行 print(stu):
#   ① 没有 __str__ → 默认输出内存地址

stu = Student("张三", 18)
print(stu)  # <__main__.Student object at 0x...>

In [ ]:
# 示例 2:添加 __str__ 后
class Student:
    def __init__(self, name, age):
        self.name = name
        self.age = age

    def __str__(self):
        # 返回友好的描述字符串
        return f"Student(name={self.name}, age={self.age})"

# --- 执行过程 ---
# 第 1 行 stu = Student("张三", 18):
#   ① 创建对象,name="张三", age=18
#
# 第 2 行 print(stu):
#   ① 自动调用 stu.__str__()
#   ② 返回 "Student(name=张三, age=18)"
#   ③ print 输出: Student(name=张三, age=18)

stu = Student("张三", 18)
print(stu)  # Student(name=张三, age=18)


**逐行解剖**

① `def __str__(self):` 定义魔术方法
② 方法名 `__str__` 前后各有**两个下划线**
③ `return f"..."` **必须返回字符串**
④ `print(stu)` 自动调用 `__str__`

**要点**:
- `__str__` 是"给人看的",用于 print 输出
- 还有一个 `__repr__` 是"给程序员看的",用于调试
- 如果只定义 `__str__`,print 用它;repr 也 fallback 用它

In [ ]:
# 示例 3:电商场景 —— Product 的 __str__
class Product:
    def __init__(self, title, price):
        self.title = title
        self.price = price

    def __str__(self):
        return f"商品[{self.title}] 价格:{self.price:.2f} 元"

# --- 执行过程 ---
# 第 1 行 p = Product("Python 入门", 59.8):
#   ① 创建对象
#
# 第 2 行 print(p):
#   ① 调用 __str__ → 返回格式化字符串
#   ② 输出: 商品[Python 入门] 价格:59.80 元

p = Product("Python 入门", 59.8)
print(p)  # 商品[Python 入门] 价格:59.80 元


**常见错误**

❌ **错误 1**:`__str__` 里用 print 而不是 return
```python
def __str__(self):
    print(...)   # 错!必须 return
    return f"..." # 对!
```

❌ **错误 2**:忘记写两个下划线
```python
def _str_(self):    # 错!单下划线
def __str__(self):  # 对!双下划线
```


> **问自己**:
> - `__str__` 什么时候被自动调用?
> - `__str__` 必须返回什么类型?
> - `__str__` 和 `__repr__` 有什么区别?

In [ ]:
# ============ 学员代码区 ============
# 请为 Student 类添加 __str__ 方法,
# 返回 "Student(name=XXX, score=XXX)"。
# 创建学生,用 print 打印它。
class Student:
    def __init__(self, name, score):
        self.name = name
        self.score = score

    # 请补全 __str__
    pass

# s = Student("小明", 85)
pass

# ============ 参考答案 ============
class Student:
    def __init__(self, name, score):
        self.name = name
        self.score = score

    def __str__(self):
        return f"Student(name={self.name}, score={self.score})"

s = Student("小明", 85)
print(s)  # Student(name=小明, score=85)

---


#### 类属性 vs 实例属性 —— 共享 vs 独立

> **痛点**  

有些数据是每个对象**独有**的(如姓名),
有些数据是**所有对象共享**的(如税率)。
如果每个对象都存一份税率,改起来太麻烦。

> **类比**  

- **实例属性** = 每个人的身份证号(各不相同)
- **类属性** = 国家的法律(所有人共享)

> **解释**  

- **类属性**:写在类体内、方法外,被所有实例**共享**
- **实例属性**:写在 `self.xxx = ...` 里,**每个实例独立**


**ASCII 内存图 —— 类属性 vs 实例属性**

```
内存布局:
  ┌────────────────────────┐
  │ Product 类              │
  │   tax_rate = 0.13      │ ← 类属性(只有一份)
  │   __init__             │
  │   price_with_tax()     │
  └──────────┬─────────────┘
             │ 引用类属性
    ┌────────┴────────┐
    ▼                 ▼
┌──────────┐    ┌──────────┐
│ p1       │    │ p2       │
│ name=Py..│    │ name=算..│ ← 实例属性
│ price=100│    │ price=200│   (各自独立)
└──────────┘    └──────────┘

Product.tax_rate = 0.09  → 所有商品都受影响!
p1.name = "新名字"       → 只影响 p1
```

**关键**:
- 类属性**一份**,所有实例**共享**
- 实例属性**每个对象一份**,互不干扰

In [ ]:
# 示例 1:类属性 vs 实例属性的基本用法
class Product:
    tax_rate = 0.13  # 类属性:所有商品共享

    def __init__(self, name, price):
        self.name = name      # 实例属性
        self.price = price    # 实例属性

    def price_with_tax(self):
        return self.price * (1 + Product.tax_rate)

# --- 执行过程 ---
# 第 1 行 p1 = Product("Python 入门", 100):
#   ① 创建对象,name="Python 入门", price=100
#
# 第 2 行 p2 = Product("算法图解", 200):
#   ① 创建对象,name="算法图解", price=200
#
# 第 3 行 print(p1.price_with_tax()):
#   ① 读取 p1.price → 100
#   ② 读取 Product.tax_rate → 0.13
#   ③ 计算 100 * 1.13 = 113.0
#
# 第 4 行 Product.tax_rate = 0.09:
#   ① 修改类属性(影响所有实例)
#
# 第 5 行 print(p1.price_with_tax()):
#   ① 读取 Product.tax_rate → 0.09(已变)
#   ② 计算 100 * 1.09 = 109.0

p1 = Product("Python 入门", 100)
p2 = Product("算法图解", 200)
print(p1.price_with_tax())  # 113.0
print(p2.price_with_tax())  # 226.0
Product.tax_rate = 0.09     # 改税率
print(p1.price_with_tax())  # 109.0(跟着变)
print(p2.price_with_tax())  # 218.0(跟着变)


**逐行解剖**

① `tax_rate = 0.13` 类属性,写在方法外
② `self.name = name` 实例属性,写在 __init__ 里
③ `Product.tax_rate` 通过**类名**访问类属性
④ `Product.tax_rate = 0.09` 修改类属性
⑤ 所有实例的 `price_with_tax()` 都受影响

**要点**:
- 类属性用 `类名.属性` 访问
- 实例属性用 `self.属性` 访问
- 修改类属性会影响**所有**实例

In [ ]:
# 示例 2:实例属性"遮蔽"类属性
class Product:
    tax_rate = 0.13

    def __init__(self, name, price):
        self.name = name
        self.price = price

# --- 执行过程 ---
# 第 1 行 p = Product("测试", 100):
#   ① 创建对象
#
# 第 2 行 p.tax_rate = 0.05:
#   ① 给 p 创建一个**实例属性** tax_rate
#   ② 实例属性"遮蔽"了类属性
#   ③ Product.tax_rate 仍然是 0.13
#
# 第 3 行 print(p.tax_rate):
#   ① 先找实例属性 → 找到 0.05
#
# 第 4 行 print(Product.tax_rate):
#   ① 直接访问类属性 → 0.13(没变)

p = Product("测试", 100)
p.tax_rate = 0.05  # 创建实例属性,不是改类属性!
print(p.tax_rate)       # 0.05(实例属性)
print(Product.tax_rate) # 0.13(类属性没变)


**常见错误**

❌ **错误 1**:通过实例修改类属性
```python
p1.tax_rate = 0.09  # 错!这是创建实例属性
Product.tax_rate = 0.09  # 对!这才是改类属性
```

❌ **错误 2**:可变类属性被实例修改
```python
class Product:
    tags = []  # 错!列表是可变的
    # 所有实例共享同一个列表,会互相干扰
```


> **问自己**:
> - 类属性和实例属性在内存中分别有几份?
> - 通过 `实例.类属性 = xxx` 会修改类属性吗?
> - 什么时候用类属性,什么时候用实例属性?

In [ ]:
# ============ 学员代码区 ============
# 请定义 Product 类:
# - 类属性 tax_rate = 0.13
# - 实例属性 name 和 price
# - 方法 price_with_tax() 返回含税价格
# 创建两个商品,改 Product.tax_rate = 0.09,
# 观察两个商品是否都跟着变。
class Product:
    pass  # 请补全

# p1 = Product("Python 入门", 100)
pass

# ============ 参考答案 ============
class Product:
    tax_rate = 0.13

    def __init__(self, name, price):
        self.name = name
        self.price = price

    def price_with_tax(self):
        return self.price * (1 + Product.tax_rate)

p1 = Product("Python 入门", 100)
p2 = Product("算法图解", 200)
Product.tax_rate = 0.09
print(p1.price_with_tax())  # 109.0
print(p2.price_with_tax())  # 218.0

---


#### 综合练习 —— BankAccount 类

现在把**六个知识点**串成一个真实业务类。

**需求清单**:
1. `__init__(owner, balance)` 绑定属性
2. `@property` 保护 balance(不允许为负)
3. 实例方法 `deposit(amount)` 存款,返回余额
4. 实例方法 `withdraw(amount)` 取款,超额拒绝
5. 类属性 `bank_name = 'Python 银行'`
6. `__str__` 输出 `BankAccount(owner=xxx, balance=xxx)`


**ASCII 内存图 —— BankAccount 完整结构**

```
  ┌──────────────────────────────┐
  │ BankAccount 类                │
  │   bank_name = "Python 银行"  │ ← 类属性
  │   __init__(self, owner, bal) │
  │   balance (property)         │
  │   balance.setter             │
  │   deposit(self, amount)      │
  │   withdraw(self, amount)     │
  │   __str__(self)              │
  └──────────────┬───────────────┘
                 │
                 ▼
          ┌──────────────┐
          │ acc          │
          │ _balance=1000│ ← 真实数据
          │ owner="张三" │
          └──────────────┘
```

**关键**:这个类包含了今天学的所有知识点,
是 OOP 封装的"最小完整案例"。

In [ ]:
# 示例:BankAccount 完整实现
class BankAccount:
    bank_name = "Python 银行"  # 类属性

    def __init__(self, owner, balance):
        self.owner = owner
        self.balance = balance  # 走 setter 校验

    @property
    def balance(self):
        return self._balance

    @balance.setter
    def balance(self, value):
        if value < 0:
            print("余额不能为负,已忽略")
            return
        self._balance = value

    def deposit(self, amount):
        self._balance += amount
        return self._balance

    def withdraw(self, amount):
        if amount > self._balance:
            print("余额不足")
            return False
        self._balance -= amount
        return True

    def __str__(self):
        return (f"BankAccount(owner={self.owner}, "
                f"balance={self._balance})")

# --- 执行过程 ---
# 第 1 行 acc = BankAccount("张三", 1000):
#   ① 创建对象,owner="张三"
#   ② self.balance = 1000 → setter → _balance=1000
#
# 第 2 行 print(acc):
#   ① 调用 __str__ → 返回格式化字符串
#
# 第 3 行 acc.deposit(500):
#   ① _balance = 1000 + 500 = 1500
#   ② 返回 1500
#
# 第 4 行 acc.withdraw(2000):
#   ① 2000 > 1500 → 打印"余额不足",返回 False
#
# 第 5 行 acc.withdraw(300):
#   ① 300 <= 1500 → _balance = 1200,返回 True

acc = BankAccount("张三", 1000)
print(acc)            # BankAccount(owner=张三, balance=1000)
print(acc.deposit(500))  # 1500
print(acc.withdraw(2000))  # 余额不足 / False
print(acc.withdraw(300))   # True
print(acc)            # BankAccount(owner=张三, balance=1200)
print(BankAccount.bank_name)  # Python 银行


**逐行解剖**

① `bank_name = "Python 银行"` 类属性,所有账户共享
② `self.balance = balance` 触发 setter 校验
③ `@property` + `@balance.setter` 保护 _balance
④ `deposit(self, amount)` 实例方法,修改余额
⑤ `withdraw(self, amount)` 实例方法,超额拒绝
⑥ `__str__` 返回友好字符串

**要点**:这个类是 OOP 封装的"最小完整案例",
包含了 class / __init__ / self / 实例方法 /
@property / __str__ / 类属性 全部知识点。


> **问自己**(综合练习前先思考):
> - 这个类用到了今天学的哪些知识点?
> - 为什么 balance 要存在 `_balance` 里?
> - 如果取款超额,应该返回什么?
> - 类属性 bank_name 应该在哪里定义?

In [ ]:
# ============ 学员代码区 ============
# 请独立完成 BankAccount 类,要求:
# 1. __init__(owner, balance) 绑定属性
# 2. @property 保护 balance(不允许为负)
# 3. deposit(amount) 存款,返回余额
# 4. withdraw(amount) 取款,超额拒绝
# 5. 类属性 bank_name = "Python 银行"
# 6. __str__ 输出友好信息
class BankAccount:
    bank_name = "Python 银行"

    def __init__(self, owner, balance):
        pass  # 请补全

    # 请补全 setter / 存取款 / __str__
    pass

# acc = BankAccount("张三", 1000)
# print(acc)
# acc.deposit(500)
# acc.withdraw(2000)
# acc.withdraw(300)
# print(acc)
pass

# ============ 参考答案 ============
class BankAccount:
    bank_name = "Python 银行"

    def __init__(self, owner, balance):
        self.owner = owner
        self.balance = balance

    @property
    def balance(self):
        return self._balance

    @balance.setter
    def balance(self, value):
        if value < 0:
            print("余额不能为负,已忽略")
            return
        self._balance = value

    def deposit(self, amount):
        self._balance += amount
        return self._balance

    def withdraw(self, amount):
        if amount > self._balance:
            print("余额不足")
            return False
        self._balance -= amount
        return True

    def __str__(self):
        return (f"BankAccount(owner={self.owner}, "
                f"balance={self._balance})")

acc = BankAccount("张三", 1000)
print(acc)               # BankAccount(owner=张三, balance=1000)
print(acc.deposit(500))  # 1500
print(acc.withdraw(2000))  # 余额不足 / False
print(acc.withdraw(300))   # True
print(acc)               # BankAccount(owner=张三, balance=1200)

---


**今日小结**

| 概念 | 解决的痛点 | 关键语法 |
| --- | --- | --- |
| 类与对象 | 把相关的数据和行为打包成一个整体 | `class` / `类名()` |
| __init__ 构造函数 | 创建对象时自动初始化属性 | `def __init__(self, ...)` |
| self 参数 | 让方法知道"当前操作的是哪个实例" | 第一个形参写 self |
| 实例方法 | 让对象自己拥有操作数据的能力 | `def method(self)` |
| @property getter | 把方法伪装成属性,统一访问语法 | `@property` |
| @property setter | 写数据时自动校验,拒绝非法值 | `@xxx.setter` |
| __str__ 魔术方法 | print(对象) 输出友好可读的信息 | `def __str__(self)` |
| 类属性 vs 实例属性 | 区分"共享数据"与"独有数据" | 类内方法外 / `self.xxx` |
| BankAccount 综合 | 把六个知识点串成一个真实业务类 | 全部综合 |

**更多练习**

- 当堂练:course/lesson08/in_class/practice01-06.py
- 课后作业:course/lesson08/homework/task01-03.py